# AI Trading Agent - Massive XGBoost Training on GPU (T4)

This notebook is designed to be run on **Google Colab** with a **T4 GPU** attached.

### Instructions:
1. Go to **Runtime -> Change runtime type** in the Colab menu.
2. Select **T4 GPU** as the hardware accelerator.
3. Click **Run All** to fetch 20 years of historical data for the top 50 NSE stocks, engineer technical features, and train a highly optimized XGBoost model.
4. Download the resulting `predictor_xgboost.pkl` file and place it in your local `ai-trading-agent/models/` folder.

In [ ]:
!pip install yfinance pandas numpy ta scikit-learn xgboost

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import ta
import pickle
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import warnings
warnings.filterwarnings('ignore')

print(f"XGBoost Version: {xgb.__version__}")

### 1. Fetching Historical Data
We will download up to 20 years of daily data for the NIFTY 50 stocks.

In [ ]:
# Top 50 NSE Stocks (NIFTY 50)
NIFTY_50 = [
    "RELIANCE.NS", "TCS.NS", "HDFCBANK.NS", "ICICIBANK.NS", "INFY.NS", 
    "SBIN.NS", "BHARTIARTL.NS", "HINDUNILVR.NS", "ITC.NS", "LT.NS", 
    "BAJFINANCE.NS", "HCLTECH.NS", "AXISBANK.NS", "KOTAKBANK.NS", "ASIANPAINT.NS", 
    "MARUTI.NS", "SUNPHARMA.NS", "TITAN.NS", "ULTRACEMCO.NS", "BAJAJFINSV.NS", 
    "WIPRO.NS", "NESTLEIND.NS", "ONGC.NS", "POWERGRID.NS", "M&M.NS", 
    "NTPC.NS", "TATASTEEL.NS", "COALINDIA.NS", "HINDALCO.NS", "TMPV.NS", 
    "JSWSTEEL.NS", "ADANIPORTS.NS", "GRASIM.NS", "ADANIENT.NS", "TECHM.NS", 
    "BRITANNIA.NS", "INDUSINDBK.NS", "BAJAJ-AUTO.NS", "EICHERMOT.NS", "CIPLA.NS", 
    "DIVISLAB.NS", "DRREDDY.NS", "HEROMOTOCO.NS", "APOLLOHOSP.NS", "HDFCLIFE.NS", 
    "SBILIFE.NS", "UPL.NS", "BPCL.NS", "SHREECEM.NS", "LTIM.NS"
]

all_data = []
for i, symbol in enumerate(NIFTY_50):
    print(f"Fetching [{i+1}/{len(NIFTY_50)}]: {symbol}")
    try:
        # Fetch 20 years of daily data
        df = yf.download(symbol, period="20y", interval="1d", progress=False)
        if not df.empty:
            # Flatten multi-index columns if present
            if isinstance(df.columns, pd.MultiIndex):
                df.columns = df.columns.droplevel(1)
            df['symbol'] = symbol
            all_data.append(df)
    except Exception as e:
        print(f"  Error fetching {symbol}: {e}")

print(f"\nFetched data for {len(all_data)} stocks.")

### 2. Feature Engineering
Calculating MACD, RSI, Bollinger Bands, ATR, and other technical indicators exactly as the agent expects them.

In [ ]:
def prepare_features(df):
    df = df.copy()
    
    # Base Indicators
    df["rsi"] = ta.momentum.RSIIndicator(df["Close"], window=14).rsi()
    df["ema_short"] = ta.trend.EMAIndicator(df["Close"], window=9).ema_indicator()
    df["ema_long"] = ta.trend.EMAIndicator(df["Close"], window=21).ema_indicator()
    df["volume_sma"] = df["Volume"].rolling(window=20).mean()
    
    # MACD
    macd = ta.trend.MACD(df["Close"])
    df["macd"] = macd.macd()
    df["macd_diff"] = macd.macd_diff()
    
    # Bollinger Bands
    bollinger = ta.volatility.BollingerBands(df["Close"])
    df["bb_high"] = bollinger.bollinger_hband()
    df["bb_low"] = bollinger.bollinger_lband()
    df["bb_mid"] = bollinger.bollinger_mavg()
    df["bb_width"] = (df["bb_high"] - df["bb_low"]) / df["bb_mid"]
    
    # ATR
    df["atr"] = ta.volatility.AverageTrueRange(df["High"], df["Low"], df["Close"]).average_true_range()
    
    # Normalized ML Features
    df["return_1d"] = df["Close"].pct_change()
    df["return_3d"] = df["Close"].pct_change(3)
    df["return_5d"] = df["Close"].pct_change(5)
    df["high_low_range"] = (df["High"] - df["Low"]) / df["Close"]
    df["close_vs_high"] = (df["Close"] - df["Low"]) / (df["High"] - df["Low"] + 1e-8)
    
    df["volatility_5d"] = df["return_1d"].rolling(5).std()
    df["volatility_20d"] = df["return_1d"].rolling(20).std()
    
    df["volume_ratio"] = df["Volume"] / df["volume_sma"]
    df["volume_change"] = df["Volume"].pct_change()
    
    df["ema_diff"] = (df["ema_short"] - df["ema_long"]) / df["Close"]
    df["price_vs_ema9"] = (df["Close"] - df["ema_short"]) / df["Close"]
    df["price_vs_ema21"] = (df["Close"] - df["ema_long"]) / df["Close"]
    df["rsi_change"] = df["rsi"].diff()
    
    df["macd_norm"] = df["macd"] / df["Close"]
    df["macd_diff_norm"] = df["macd_diff"] / df["Close"]
    df["price_vs_bb_mid"] = (df["Close"] - df["bb_mid"]) / df["Close"]
    df["atr_norm"] = df["atr"] / df["Close"]

    # Target: Will price go up by > 0.3% tomorrow?
    df["target"] = (df["Close"].shift(-1) > df["Close"] * 1.003).astype(int)
    
    return df

FEATURE_COLS = [
    "rsi", "rsi_change", "ema_diff", "price_vs_ema9", "price_vs_ema21",
    "return_1d", "return_3d", "return_5d",
    "high_low_range", "close_vs_high",
    "volatility_5d", "volatility_20d",
    "volume_ratio", "volume_change",
    "macd_norm", "macd_diff_norm", "bb_width", "price_vs_bb_mid", "atr_norm"
]

processed_data = []
for df in all_data:
    features = prepare_features(df)
    processed_data.append(features)

combined = pd.concat(processed_data, ignore_index=True)
combined = combined.dropna(subset=FEATURE_COLS + ["target"])
combined = combined.replace([np.inf, -np.inf], np.nan).dropna(subset=FEATURE_COLS)

print(f"Total training samples prepared: {len(combined):,}")

### 3. GPU-Accelerated Training with XGBoost
Using the `hist` tree method with `device='cuda'` enables the T4 GPU to train on millions of rows in seconds.

In [ ]:
X = combined[FEATURE_COLS].values
y = combined["target"].values

# Split into Train (80%) and Test (20%) sequentially to prevent time-series data leakage
split_idx = int(len(X) * 0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

# Calculate scale_pos_weight to handle class imbalance
sum_wpos = sum(y_train == 1)
sum_wneg = sum(y_train == 0)
scale_pos_weight = sum_wneg / sum_wpos if sum_wpos > 0 else 1.0

print("Training XGBoost model on GPU...")

# Configure XGBoost for GPU
clf = xgb.XGBClassifier(
    n_estimators=2000,
    max_depth=6,
    learning_rate=0.01,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    early_stopping_rounds=50,
    tree_method='hist',
    device='cuda',      # Critical: This tells XGBoost to use the T4 GPU
    random_state=42,
    n_jobs=-1
)

clf.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=100)

y_pred = clf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"\nTest Accuracy: {accuracy * 100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

### 4. Save the Model
We save the model using pickle so it's directly compatible with the `ai-trading-agent`'s `predictor.py`.

In [ ]:
model_path = "predictor_xgboost.pkl"
with open(model_path, "wb") as f:
    pickle.dump(clf, f)

print(f"Model saved successfully to {model_path}!")
print("You can now download this file from the Colab file explorer on the left and place it in your local 'models' folder.")